# 発展のノート（任意）

実習①・②のあとに、もっと見たい人のためのノートです。コードの中身は読み飛ばし、出てくるグラフだけ見てもかまいません。

- 使い方は実習①・②と同じです。上のセルから順に、全部押してください。
- このノートだけで動きます（実習①・②のノートを先に押しておく必要はありません）。
- 自分の動画でSAM-Body4Dを動かしたいときは、推論用のノート`00_body4d_inference.ipynb`を使います（GPUとHugging Faceの準備が必要）。

## 1. 道具を読み込む

In [ ]:
!pip install -q matplotlib-fontja
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_fontja
print('道具の準備ができました')

## 2. データを持ってくる

実習①・②と同じIDを、それぞれ`'`と`'`の間に貼り付けてから▶を押してください。

In [ ]:
ROM_FILE_ID = 'YOUR_ROM_FILE_ID_HERE'
REACH_FILE_ID = 'YOUR_REACH_FILE_ID_HERE'
!gdown {ROM_FILE_ID} -O rom_task_output.json
!gdown {REACH_FILE_ID} -O reaching_output.json

## 3. 角度を出す（実習①と同じ関数）

実習①で作った関数を、もう一度作って全コマに当てます。

In [ ]:
with open('rom_task_output.json') as f:
    data = json.load(f)
joints = np.array(data['joints'])
fps = data['meta']['fps']
n_frames = len(joints)
time = np.arange(n_frames) / fps

def angle_between_vectors(v1, v2):
    v1 = v1 / np.linalg.norm(v1)
    v2 = v2 / np.linalg.norm(v2)
    cos_angle = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return np.degrees(np.arccos(cos_angle))

def compute_shoulder_elevation(frame, side='right'):
    if side == 'right':
        shoulder, elbow = frame[17], frame[19]
    else:
        shoulder, elbow = frame[16], frame[18]
    return angle_between_vectors(elbow - shoulder, frame[0] - frame[12])

def compute_elbow_flexion(frame, side='right'):
    if side == 'right':
        shoulder, elbow, wrist = frame[17], frame[19], frame[21]
    else:
        shoulder, elbow, wrist = frame[16], frame[18], frame[20]
    inner = angle_between_vectors(shoulder - elbow, wrist - elbow)
    return 180.0 - inner

side = 'right'
shoulder_elev = np.array([compute_shoulder_elevation(f, side) for f in joints])
elbow_flex = np.array([compute_elbow_flexion(f, side) for f in joints])
print('肩挙上の最大', round(shoulder_elev.max(), 1))
print('肘屈曲の最大', round(elbow_flex.max(), 1))

## 4. 体幹の代償

挙上に伴って、体幹（骨盤→首の線）が開始姿勢からどれだけ傾いたかを測ります。開始の5コマの平均を基準にします。

In [ ]:
def trunk_lean_series(joints, baseline_frames=5):
    trunk = joints[:, 12, :] - joints[:, 0, :]   # 骨盤→首
    base = trunk[:baseline_frames].mean(axis=0)
    return np.array([angle_between_vectors(t, base) for t in trunk])

trunk_lean = trunk_lean_series(joints)
print('体幹の傾きの最大', round(trunk_lean.max(), 1))

plt.plot(time, trunk_lean, label='体幹の傾き')
plt.axhline(5, color='gray', linestyle='--', label='代償の目安 5°')
plt.xlabel('時間 [秒]')
plt.ylabel('体幹の傾き [度]')
plt.legend()
plt.show()

## 5. 肩と肘の協調

横軸に肩挙上角、縦軸に肘屈曲角をとって、点を時間の順に結びます（angle-angle図）。上げるときと下ろすときで同じ道を通るかを見ます。

In [ ]:
r = np.corrcoef(shoulder_elev, elbow_flex)[0, 1]
print('肩挙上と肘屈曲の相関 r', round(r, 2))

plt.plot(shoulder_elev, elbow_flex, color='gray', alpha=0.5)
plt.scatter(shoulder_elev, elbow_flex, c=time, s=15)
plt.colorbar(label='時間 [秒]')
plt.xlabel('肩挙上角 [度]')
plt.ylabel('肘屈曲角 [度]')
plt.show()

## 6. 手首の軌跡を立体で見る

実習②の注ぐ動作で、右手首が通った道を立体で描きます。色は時間（濃い色が始め）です。静止画なので、マウスで回すことはできません。

In [ ]:
with open('reaching_output.json') as f:
    reach = json.load(f)
wrist = np.array(reach['joints'])[:, 21]
rel = wrist - wrist[0]
ax = plt.figure(figsize=(7, 6)).add_subplot(projection='3d')
ax.scatter(rel[:, 0], rel[:, 1], rel[:, 2],
           c=np.arange(len(rel)), cmap='plasma', s=10)
ax.set_xlabel('横 [m]')
ax.set_ylabel('上下 [m]')
ax.set_zlabel('前後 [m]')
plt.show()

## 7. 考えてみる

1. 体幹の傾きは、挙上のどの時点で大きくなるか。5°を超える場面はあるか。
2. angle-angle図で、上げるときと下ろすときの道は重なるか。
3. 手首の軌跡は、注ぐ・手に取るのどの場面で大きく動いているか。